# Deep Learning - Project 1

**Name:** `Robert-Constantin Bujor`

**NetID:** `rcb9438`

## Training on Kaggle (2× T4)

NYU HPC GPUs were not available during the project, so the 60- and 200-epoch comparisons were trained
in this Kaggle notebook. It writes the repository's source files to disk and runs `train.py` as a
normal program, one queue of runs per GPU.

**Before running**
1. Right panel -> **Session options** -> **Accelerator** -> **GPU T4 x2**.
2. Right panel -> **Input** -> **Add Input** -> **Competitions** -> the course competition.
3. **Save Version** -> **Save & Run All (Commit)** runs everything in the background
   (about 1-1.5 hours for a 60-epoch batch) and the tab can be closed. **Run All** shows it live instead.

When it finishes, the **Output** tab has `submission.csv` (best run), one `submission_<run>.csv` per run,
`runs/summary.csv` and the training logs.

In [ ]:
import glob
import os
import shutil
import subprocess
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch


WORK_DIR = "/kaggle/working/cifar-resnet"

n_gpus = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(n_gpus)])

if n_gpus == 0:
    raise RuntimeError("No GPU: set Accelerator to GPU T4 x2")

found = glob.glob("/kaggle/input/**/train_labels.csv", recursive=True)

if not found:
    raise FileNotFoundError(
        "Competition data not found: Add Input -> Competitions -> the course competition"
    )

SOURCE_DIR = os.path.dirname(found[0])
print("competition data:", SOURCE_DIR, sorted(os.listdir(SOURCE_DIR)))

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

## Source files

The same files as in the repository, written to disk so the training script runs as a normal
program.

In [ ]:
%%writefile models.py
"""Residual networks for 32 x 32 CIFAR-10 images.

Two residual blocks are defined:

- BasicBlock is the block from the project handout,
  out = ReLU(S(x) + F(x)) with F = conv -> BN -> ReLU -> conv -> BN.
- PreActBlock is the pre-activation block of He et al. (2016), used by Wide
  ResNets: out = S(x) + conv(ReLU(BN(conv(ReLU(BN(x)))))).

A network is a 3 x 3 stem convolution, a few stages of residual blocks
(every stage after the first halves the resolution), global average pooling
and a linear classifier. Each architecture from the experiments is one entry
in CONFIGS, so an experiment only needs a different --model name.

Run `python models.py` to print the parameter count of each config.
"""
import torch.nn as nn
import torch.nn.functional as F


PARAM_LIMIT = 5_000_000


class BasicBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1,
        kernel=3,
        skip_kernel=1,
        dropout=0.0,
    ):
        super().__init__()
        padding = kernel // 2

        self.conv1 = nn.Conv2d(
            in_channels, out_channels, kernel, stride, padding, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(
            out_channels, out_channels, kernel, 1, padding, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        # When the block changes the number of channels or the resolution,
        # the shortcut needs a projection so it can be added to F(x).
        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    skip_kernel,
                    stride,
                    skip_kernel // 2,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(self.dropout(out)))
        return F.relu(out + self.skip(x))


class PreActBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1,
        kernel=3,
        skip_kernel=1,
        dropout=0.0,
    ):
        super().__init__()
        padding = kernel // 2

        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, kernel, stride, padding, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(
            out_channels, out_channels, kernel, 1, padding, bias=False
        )
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.projection = stride != 1 or in_channels != out_channels

        if self.projection:
            self.skip = nn.Conv2d(
                in_channels,
                out_channels,
                skip_kernel,
                stride,
                skip_kernel // 2,
                bias=False,
            )

    def forward(self, x):
        pre = F.relu(self.bn1(x))

        # The projection shortcut also starts from the normalized input,
        # while the identity shortcut keeps x unchanged.
        shortcut = self.skip(pre) if self.projection else x

        out = self.conv1(pre)
        out = self.conv2(self.dropout(F.relu(self.bn2(out))))
        return out + shortcut


BLOCKS = {"basic": BasicBlock, "preact": PreActBlock}


class ResNet(nn.Module):
    def __init__(
        self,
        widths,
        depths,
        block="basic",
        stem_width=None,
        kernel=3,
        skip_kernel=1,
        dropout=0.0,
        num_classes=10,
        zero_init_residual=True,
    ):
        super().__init__()

        if len(widths) != len(depths):
            raise ValueError("widths and depths need one entry per stage.")

        block_class = BLOCKS[block]
        self.preact = block == "preact"
        stem_width = stem_width or widths[0]

        self.stem = nn.Conv2d(3, stem_width, 3, 1, 1, bias=False)

        # A post-activation network normalizes right after the stem, while a
        # pre-activation network does it at the start of its first block.
        if self.preact:
            self.stem_bn = nn.Identity()
        else:
            self.stem_bn = nn.BatchNorm2d(stem_width)

        stages = []
        in_channels = stem_width

        for stage_index, (width, depth) in enumerate(zip(widths, depths)):
            blocks = []

            for block_index in range(depth):
                # The first block of every stage after the first one halves
                # the resolution.
                stride = 2 if stage_index > 0 and block_index == 0 else 1
                blocks.append(
                    block_class(
                        in_channels, width, stride, kernel, skip_kernel, dropout
                    )
                )
                in_channels = width

            stages.append(nn.Sequential(*blocks))

        self.stages = nn.Sequential(*stages)

        if self.preact:
            self.final_bn = nn.BatchNorm2d(in_channels)
        else:
            self.final_bn = nn.Identity()

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(in_channels, num_classes)
        self.initialize_weights(zero_init_residual)

    def initialize_weights(self, zero_init_residual):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(
                    module.weight, mode="fan_out", nonlinearity="relu"
                )
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

        if zero_init_residual:
            # The last BN of every basic block starts at zero, so each block
            # begins as the identity (Goyal et al., 2017), which makes the
            # first epochs more stable. A pre-activation block ends in a
            # convolution instead of a BN, so it is left as it is.
            for module in self.modules():
                if isinstance(module, BasicBlock):
                    nn.init.zeros_(module.bn2.weight)

    def forward(self, x):
        x = self.stem(x)

        if not self.preact:
            x = F.relu(self.stem_bn(x))

        x = self.stages(x)

        if self.preact:
            x = F.relu(self.final_bn(x))

        x = self.pool(x).flatten(1)
        return self.fc(x)


CONFIGS = {
    # ResNet-18 layout (4 stages x 2 blocks) with a slimmer last stage.
    "resnet18_slim": {
        "block": "basic",
        "widths": (64, 128, 256, 224),
        "depths": (2, 2, 2, 2),
    },
    # Shallower but wider: 3 stages x 2 blocks.
    "resnet14_wide": {
        "block": "basic",
        "widths": (96, 192, 320),
        "depths": (2, 2, 2),
    },
    # Wide ResNet with depth 28 and widening factor 3.5
    # (Zagoruyko & Komodakis, 2016). This is the submitted model.
    "wrn28_3.5": {
        "block": "preact",
        "widths": (56, 112, 224),
        "depths": (4, 4, 4),
        "stem_width": 16,
    },
    # Teacher for distillation only. It is far above the 5M limit and was
    # never submitted.
    "wrn28_10": {
        "block": "preact",
        "widths": (160, 320, 640),
        "depths": (4, 4, 4),
        "stem_width": 16,
    },
    # The original ResNet-18 widths (11.2M parameters), for reference only.
    "resnet18": {
        "block": "basic",
        "widths": (64, 128, 256, 512),
        "depths": (2, 2, 2, 2),
    },
}


def build_model(name, **overrides):
    if name not in CONFIGS:
        raise ValueError(f"unknown model '{name}', choose from {sorted(CONFIGS)}")

    settings = dict(CONFIGS[name])
    settings.update(overrides)
    return ResNet(**settings)


def count_params(model):
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


if __name__ == "__main__":
    for name in CONFIGS:
        n_params = count_params(build_model(name))
        status = "ok" if n_params <= PARAM_LIMIT else "OVER LIMIT"
        print(f"{name:15s} {n_params:>11,d}  {status}")

In [ ]:
%%writefile data.py
"""Loading the CIFAR-10 arrays and building the training augmentations.

All images stay in memory as one uint8 tensor of shape (N, 3, 32, 32).
Per-image augmentation runs inside the DataLoader workers, while mixup and
CutMix work on whole batches on the GPU and live in train.py.
"""
import os

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2


AUG_CHOICES = {"cutout", "ta", "aa", "ra"}
NUM_CLASSES = 10


def load_arrays(data_dir="data", val_size=5000, seed=0):
    # Returns the training split, the validation split (None when
    # val_size is 0), the test images with their ids, and the per-channel
    # mean and std used for normalization.
    train = np.load(os.path.join(data_dir, "train.npz"))
    test = np.load(os.path.join(data_dir, "test.npz"))

    images = torch.from_numpy(train["images"]).permute(0, 3, 1, 2).contiguous()
    labels = torch.from_numpy(train["labels"])
    stats = (tuple(train["mean"].tolist()), tuple(train["std"].tolist()))

    if val_size > 0:
        # The same number of images is held out from every class. The split
        # only depends on the seed, so runs with the same seed share it.
        rng = np.random.default_rng(seed)
        per_class = val_size // NUM_CLASSES
        val_index = np.concatenate([
            rng.choice(
                np.flatnonzero(train["labels"] == label),
                per_class,
                replace=False,
            )
            for label in range(NUM_CLASSES)
        ])

        keep = np.ones(len(labels), dtype=bool)
        keep[val_index] = False
        train_index = np.flatnonzero(keep)

        train_split = (images[train_index], labels[train_index])
        val_split = (images[val_index], labels[val_index])
    else:
        train_split = (images, labels)
        val_split = None

    test_images = torch.from_numpy(test["images"]).permute(0, 3, 1, 2).contiguous()
    test_ids = torch.from_numpy(test["ids"])

    return train_split, val_split, (test_images, test_ids), stats


class Cutout:
    # Cutout (DeVries & Taylor, 2017) sets one size x size square at a random
    # centre to zero. The square can hang over the border, so the erased area
    # varies. It runs after normalization, where 0 is the dataset mean.

    def __init__(self, size=16):
        self.size = size

    def __call__(self, image):
        _, height, width = image.shape
        centre_y = int(torch.randint(height, (1,)))
        centre_x = int(torch.randint(width, (1,)))
        half = self.size // 2

        top, bottom = max(centre_y - half, 0), min(centre_y + half, height)
        left, right = max(centre_x - half, 0), min(centre_x + half, width)

        image = image.clone()
        image[:, top:bottom, left:right] = 0.0
        return image


def parse_aug(aug):
    if not aug:
        return set()

    # Crop and flip are always used, so "basic" adds nothing to the set.
    parts = {part for part in aug.split("+") if part and part != "basic"}
    unknown = parts - AUG_CHOICES

    if unknown:
        raise ValueError(
            f"unknown augmentation(s) {unknown}; choose from {AUG_CHOICES}"
        )

    return parts


def train_transform(aug, mean, std, cutout_size=16):
    # aug is a "+"-separated list such as "basic", "cutout" or "aa+cutout":
    #   cutout  Cutout with a 16 x 16 square
    #   ta      TrivialAugmentWide (Mueller & Hutter, 2021)
    #   aa      AutoAugment with the CIFAR-10 policy (Cubuk et al., 2019)
    #   ra      RandAugment (Cubuk et al., 2020)
    # A random crop with 4-pixel reflect padding and a horizontal flip are
    # always applied first.
    parts = parse_aug(aug)
    steps = [
        v2.RandomCrop(32, padding=4, padding_mode="reflect"),
        v2.RandomHorizontalFlip(),
    ]

    if "ta" in parts:
        steps.append(v2.TrivialAugmentWide())

    if "aa" in parts:
        steps.append(v2.AutoAugment(v2.AutoAugmentPolicy.CIFAR10))

    if "ra" in parts:
        steps.append(v2.RandAugment())

    steps.append(v2.ToDtype(torch.float32, scale=True))
    steps.append(v2.Normalize(mean, std))

    if "cutout" in parts:
        steps.append(Cutout(cutout_size))

    return v2.Compose(steps)


class ImageTensorDataset(Dataset):
    def __init__(self, images, targets, transform):
        self.images = images
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        return self.transform(self.images[index]), self.targets[index]


def make_loader(images, targets, transform, batch_size, train, workers):
    dataset = ImageTensorDataset(images, targets, transform)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=train,
        drop_last=train,
        num_workers=workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=workers > 0,
        prefetch_factor=4 if workers > 0 else None,
    )

In [ ]:
%%writefile utils.py
"""Helpers shared by train.py, predict.py and the notebooks."""
import contextlib
import csv

import torch
import torch.nn.functional as F

from models import build_model


def get_device():
    # An NVIDIA GPU if there is one, then the Apple-silicon GPU (MPS),
    # otherwise the CPU.
    if torch.cuda.is_available():
        return torch.device("cuda")

    if torch.backends.mps.is_available():
        return torch.device("mps")

    return torch.device("cpu")


def device_name(device):
    if device.type == "cuda":
        return torch.cuda.get_device_name()

    if device.type == "mps":
        return "Apple GPU (MPS)"

    return "cpu"


def memory_format(device):
    # channels_last speeds up convolutions on NVIDIA tensor cores. Other
    # devices keep the default layout.
    if device.type == "cuda":
        return torch.channels_last

    return torch.contiguous_format


def amp_dtype(device, enabled=True):
    # Mixed precision is only used on NVIDIA GPUs: bfloat16 on Ampere or
    # newer (A100, L4, RTX 30/40), float16 on older cards (V100, T4).
    # None means full float32.
    if not enabled or device.type != "cuda":
        return None

    major, _ = torch.cuda.get_device_capability()
    return torch.bfloat16 if major >= 8 else torch.float16


def autocast(device, dtype):
    if dtype is None:
        return contextlib.nullcontext()

    return torch.autocast(device.type, dtype=dtype)


@torch.no_grad()
def predict_logprobs(
    model,
    images,
    mean,
    std,
    device,
    tta=True,
    batch_size=1000,
    dtype=None,
):
    # images is a uint8 tensor of shape (N, 3, 32, 32). With tta=True the
    # class probabilities of each image and of its mirror image are averaged
    # (horizontal-flip test-time augmentation).
    model.eval()
    mean_tensor = torch.tensor(mean, device=device).view(1, 3, 1, 1)
    std_tensor = torch.tensor(std, device=device).view(1, 3, 1, 1)
    chunks = []

    for start in range(0, len(images), batch_size):
        batch = images[start:start + batch_size].to(device).float().div_(255)
        batch = ((batch - mean_tensor) / std_tensor).contiguous(
            memory_format=memory_format(device)
        )

        with autocast(device, dtype):
            probabilities = F.softmax(model(batch).float(), dim=1)

            if tta:
                flipped = torch.flip(batch, dims=[3])
                flipped_probabilities = F.softmax(model(flipped).float(), dim=1)
                probabilities = (probabilities + flipped_probabilities) / 2

        chunks.append(probabilities.clamp_min(1e-12).log().cpu())

    return torch.cat(chunks)


def accuracy(logprobs, labels):
    return (logprobs.argmax(1) == labels).float().mean().item()


def write_submission(path, ids, predictions):
    # Kaggle format: an "id,label" header and one row per test image,
    # sorted by id.
    rows = sorted(zip(ids.tolist(), predictions.tolist()))

    with open(path, "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["id", "label"])
        writer.writerows(rows)


def load_trained_model(path, device, which=None):
    # Rebuilds a model saved by train.py. which is "ema", "raw" or None,
    # where None loads the weights train.py selected for the submission.
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    model = build_model(
        checkpoint["model"],
        **checkpoint.get("model_overrides", {}),
    )

    which = which or checkpoint.get("selected", "raw")

    if which == "ema":
        model.load_state_dict(checkpoint["ema_state_dict"])
    else:
        model.load_state_dict(checkpoint["state_dict"])

    model.to(device).to(memory_format=memory_format(device)).eval()
    return model, checkpoint

In [ ]:
%%writefile train.py
"""Train a ResNet with at most 5M parameters and write a Kaggle submission.

Everything a run produces goes to runs/<name>/:
    config.json      all arguments
    log.csv          learning rate, losses and accuracies for every epoch
    model.pt         final weights and EMA weights (utils.load_trained_model)
    metrics.json     final numbers
    submission.csv   test predictions in the Kaggle format
and one summary row is added to runs/summary.csv.

A resumable checkpoint (last.pt) is written every --save-every epochs. If a
run stops early, run the same command again with --resume added.

Examples:
    python train.py --name baseline --model resnet18_slim --epochs 30
    python train.py --name final --model wrn28_3.5 --epochs 300 --aug aa+cutout --val-size 0
    python train.py --name final --model wrn28_3.5 --epochs 300 --aug aa+cutout --val-size 0 --resume
"""
import argparse
import copy
import csv
import json
import math
import os
import random
import time

# On Apple silicon, the few operations without an MPS kernel fall back to
# the CPU instead of failing.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from data import load_arrays, make_loader, parse_aug, train_transform
from models import CONFIGS, PARAM_LIMIT, build_model, count_params
from utils import (
    accuracy,
    amp_dtype,
    autocast,
    device_name,
    get_device,
    load_trained_model,
    memory_format,
    predict_logprobs,
    write_submission,
)


LOG_FIELDS = [
    "epoch",
    "lr",
    "train_loss",
    "train_acc",
    "val_loss",
    "val_acc",
    "ema_val_acc",
    "epoch_time",
]


def get_args(argv=None):
    parser = argparse.ArgumentParser(
        description="Train a CIFAR ResNet",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )

    # Run bookkeeping
    parser.add_argument(
        "--name", required=True, help="run name; outputs go to <out-dir>/<name>"
    )
    parser.add_argument("--out-dir", default="runs")
    parser.add_argument("--data-dir", default="data")
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument(
        "--workers",
        type=int,
        default=None,
        help="DataLoader workers (default: $SLURM_CPUS_PER_TASK or 4)",
    )

    # Model
    parser.add_argument("--model", default="resnet18_slim", choices=sorted(CONFIGS))
    parser.add_argument("--dropout", type=float, default=0.0)
    parser.add_argument(
        "--allow-over-limit",
        action="store_true",
        help="allow > 5M parameters (only for distillation teachers)",
    )

    # Data
    parser.add_argument(
        "--val-size",
        type=int,
        default=5000,
        help="images held out for validation; 0 = train on all 50k",
    )
    parser.add_argument(
        "--aug", default="cutout", help="'+'-separated: basic, cutout, ta, aa, ra"
    )
    parser.add_argument(
        "--mix", default="none", choices=["none", "mixup", "cutmix", "both"]
    )
    parser.add_argument("--mix-alpha", type=float, default=1.0)
    parser.add_argument(
        "--mix-prob",
        type=float,
        default=0.5,
        help="fraction of batches that get mixed",
    )

    # Optimization
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=0.1, help="peak learning rate")
    parser.add_argument("--momentum", type=float, default=0.9)
    parser.add_argument(
        "--wd",
        type=float,
        default=5e-4,
        help="weight decay (not applied to BN/bias)",
    )
    parser.add_argument("--warmup-epochs", type=float, default=2)
    parser.add_argument("--label-smoothing", type=float, default=0.1)
    parser.add_argument(
        "--ema",
        type=float,
        default=0.999,
        help="EMA decay per step; 0 disables EMA",
    )

    # Knowledge distillation
    parser.add_argument(
        "--teacher",
        default=None,
        help="teacher checkpoint (runs/<teacher>/model.pt)",
    )
    parser.add_argument(
        "--kd-alpha",
        type=float,
        default=0.5,
        help="weight of the distillation loss",
    )
    parser.add_argument(
        "--kd-temp", type=float, default=4.0, help="softmax temperature"
    )

    # Speed and debugging
    parser.add_argument(
        "--no-amp", action="store_true", help="disable mixed precision"
    )
    parser.add_argument("--compile", action="store_true", help="use torch.compile")
    parser.add_argument(
        "--limit-batches",
        type=int,
        default=0,
        help="debug: batches per epoch",
    )

    # Checkpointing
    parser.add_argument(
        "--save-every",
        type=int,
        default=5,
        help="save a resumable checkpoint (last.pt) every N epochs; 0 disables",
    )
    parser.add_argument(
        "--resume",
        action="store_true",
        help="continue an interrupted run from <out-dir>/<name>/last.pt",
    )

    return parser.parse_args(argv)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


class EMA:
    # Exponential moving average of the weights and the BatchNorm
    # statistics. The averaged model is usually a little more accurate than
    # the raw one.

    def __init__(self, model, decay):
        self.decay = decay
        self.module = copy.deepcopy(model).eval()

        for parameter in self.module.parameters():
            parameter.requires_grad_(False)

        self.updates = 0

    @torch.no_grad()
    def update(self, model):
        self.updates += 1

        # A short warm-up, so the random starting weights are forgotten
        # quickly.
        decay = min(self.decay, (1 + self.updates) / (10 + self.updates))
        current = model.state_dict()

        for key, average in self.module.state_dict().items():
            if average.dtype.is_floating_point:
                average.lerp_(current[key].detach(), 1 - decay)
            else:
                average.copy_(current[key])


def mix_batch(images, labels, mode, alpha):
    # Mixup (Zhang et al., 2018) or CutMix (Yun et al., 2019) on one batch.
    # Returns the mixed images, both sets of labels and the weight of the
    # first set.
    if mode == "both":
        mode = "mixup" if random.random() < 0.5 else "cutmix"

    lam = float(np.random.beta(alpha, alpha))
    permutation = torch.randperm(images.size(0), device=images.device)

    if mode == "mixup":
        images = lam * images + (1 - lam) * images[permutation]
    else:
        height, width = images.shape[-2:]
        ratio = math.sqrt(1 - lam)
        cut_height, cut_width = int(height * ratio), int(width * ratio)
        centre_y, centre_x = random.randrange(height), random.randrange(width)

        top = max(centre_y - cut_height // 2, 0)
        bottom = min(centre_y + cut_height // 2, height)
        left = max(centre_x - cut_width // 2, 0)
        right = min(centre_x + cut_width // 2, width)

        images = images.clone()
        images[:, :, top:bottom, left:right] = (
            images[permutation, :, top:bottom, left:right]
        )

        # The patch is clipped at the border, so the label weight follows
        # the area that was actually pasted.
        lam = 1 - (bottom - top) * (right - left) / (height * width)

    return images, labels, labels[permutation], lam


def distillation_loss(student_logits, teacher_logits, temperature):
    # KL divergence between the softened teacher and student distributions
    # (Hinton et al., 2015). Multiplying by T^2 keeps the gradients about as
    # large as those of the cross-entropy term.
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
    teacher_log_probs = F.log_softmax(teacher_logits / temperature, dim=1)

    divergence = F.kl_div(
        student_log_probs,
        teacher_log_probs,
        reduction="batchmean",
        log_target=True,
    )
    return divergence * temperature * temperature


def make_optimizer(model, lr, momentum, weight_decay):
    # Weight decay is only applied to convolution and linear weights.
    # BatchNorm weights and biases are 1-D, so they are left out.
    decay = []
    no_decay = []

    for parameter in model.parameters():
        if parameter.ndim > 1:
            decay.append(parameter)
        else:
            no_decay.append(parameter)

    groups = [
        {"params": decay, "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]
    return torch.optim.SGD(groups, lr=lr, momentum=momentum, nesterov=True)


def make_scheduler(optimizer, total_steps, warmup_steps):
    # Linear warm-up, then cosine decay to zero, updated after every step.
    def factor(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps

        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


def save_checkpoint(path, epoch, model, ema, optimizer, scheduler, scaler, minutes):
    # Everything needed to continue exactly where the run stopped.
    state = {
        "epoch": epoch,
        "model": model.state_dict(),
        "ema": ema.module.state_dict() if ema is not None else None,
        "ema_updates": ema.updates if ema is not None else 0,
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "rng_python": random.getstate(),
        "rng_numpy": np.random.get_state(),
        "rng_torch": torch.get_rng_state(),
        "minutes": minutes,
    }

    # The checkpoint is written to a temporary file first, so stopping the
    # run in the middle of a save never leaves a broken last.pt.
    torch.save(state, path + ".tmp")
    os.replace(path + ".tmp", path)


def load_checkpoint(path, device, model, ema, optimizer, scheduler, scaler, log_path):
    state = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(state["model"])

    if ema is not None and state["ema"] is not None:
        ema.module.load_state_dict(state["ema"])
        ema.updates = state["ema_updates"]

    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    scaler.load_state_dict(state["scaler"])
    random.setstate(state["rng_python"])
    np.random.set_state(state["rng_numpy"])
    torch.set_rng_state(state["rng_torch"].cpu())

    # Log rows written after the checkpoint are dropped, because those
    # epochs are trained again.
    with open(log_path, newline="") as file:
        kept_rows = [
            row
            for row in csv.DictReader(file)
            if int(row["epoch"]) <= state["epoch"]
        ]

    with open(log_path, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=LOG_FIELDS)
        writer.writeheader()
        writer.writerows(kept_rows)

    return state["epoch"], state["minutes"]


def train_one_epoch(
    args,
    train_model,
    model,
    ema,
    teacher,
    loader,
    optimizer,
    scheduler,
    scaler,
    criterion,
    device,
    dtype,
):
    train_model.train()
    loss_sum = torch.zeros((), device=device)
    correct_sum = torch.zeros((), device=device)
    seen = 0

    for step, (images, labels) in enumerate(loader):
        if args.limit_batches and step >= args.limit_batches:
            break

        images = images.to(device, non_blocking=True).contiguous(
            memory_format=memory_format(device)
        )
        labels = labels.to(device, non_blocking=True)

        # Only a fraction of the batches (--mix-prob) is mixed.
        if args.mix != "none" and random.random() < args.mix_prob:
            images, labels_a, labels_b, lam = mix_batch(
                images, labels, args.mix, args.mix_alpha
            )
        else:
            labels_a, labels_b, lam = labels, labels, 1.0

        with autocast(device, dtype):
            logits = train_model(images)
            loss = (
                lam * criterion(logits, labels_a)
                + (1 - lam) * criterion(logits, labels_b)
            )

            if teacher is not None:
                # The teacher sees the same augmented (and mixed) batch.
                with torch.no_grad():
                    teacher_logits = teacher(images)

                kd_loss = distillation_loss(
                    logits.float(), teacher_logits.float(), args.kd_temp
                )
                loss = (1 - args.kd_alpha) * loss + args.kd_alpha * kd_loss

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        if ema is not None:
            ema.update(model)

        predictions = logits.argmax(1)
        loss_sum += loss.detach() * len(labels)
        correct_sum += (
            lam * (predictions == labels_a).sum()
            + (1 - lam) * (predictions == labels_b).sum()
        )
        seen += len(labels)

    return loss_sum / seen, correct_sum / seen


def append_summary(path, row):
    new_file = not os.path.exists(path)

    with open(path, "a", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(row))

        if new_file:
            writer.writeheader()

        writer.writerow(row)


def main(args):
    seed_everything(args.seed)
    device = get_device()
    dtype = amp_dtype(device, not args.no_amp)
    torch.backends.cudnn.benchmark = True

    run_dir = os.path.join(args.out_dir, args.name)
    os.makedirs(run_dir, exist_ok=True)

    if args.workers is not None:
        workers = args.workers
    else:
        # On a Slurm cluster, use as many workers as the job has CPUs.
        workers = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))

    # Checking the augmentation name first means a typo fails right away
    # instead of after the data is loaded.
    parse_aug(args.aug)

    # Data
    train_split, val_split, (test_images, test_ids), (mean, std) = load_arrays(
        args.data_dir, args.val_size, args.seed
    )
    train_images, train_labels = train_split
    train_loader = make_loader(
        train_images,
        train_labels,
        train_transform(args.aug, mean, std),
        args.batch_size,
        train=True,
        workers=workers,
    )

    # Model
    overrides = {"dropout": args.dropout} if args.dropout else {}
    model = build_model(args.model, **overrides)
    model = model.to(device).to(memory_format=memory_format(device))
    n_params = count_params(model)

    if n_params > PARAM_LIMIT and not args.allow_over_limit:
        raise SystemExit(
            f"{args.model} has {n_params:,} parameters (> {PARAM_LIMIT:,})."
        )

    train_model = torch.compile(model) if args.compile else model
    ema = EMA(model, args.ema) if args.ema > 0 else None

    teacher = None

    if args.teacher:
        teacher, _ = load_trained_model(args.teacher, device)
        print(f"teacher: {args.teacher} ({count_params(teacher):,} params)")

    gpu = device_name(device)
    val_count = 0 if val_split is None else len(val_split[0])
    print(
        f"model {args.model}: {n_params:,} trainable parameters | device {gpu} | "
        f"amp {dtype} | train {len(train_images)} val {val_count} | "
        f"workers {workers}",
        flush=True,
    )

    with open(os.path.join(run_dir, "config.json"), "w") as file:
        json.dump({**vars(args), "params": n_params, "gpu": gpu}, file, indent=2)

    # Optimization
    steps_per_epoch = args.limit_batches or len(train_loader)
    optimizer = make_optimizer(model, args.lr, args.momentum, args.wd)
    scheduler = make_scheduler(
        optimizer,
        args.epochs * steps_per_epoch,
        int(args.warmup_epochs * steps_per_epoch),
    )
    scaler = torch.amp.GradScaler(enabled=dtype == torch.float16)
    criterion = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)

    log_path = os.path.join(run_dir, "log.csv")
    last_path = os.path.join(run_dir, "last.pt")
    first_epoch = 1
    minutes_before = 0.0

    if args.resume and os.path.exists(last_path):
        saved_epoch, minutes_before = load_checkpoint(
            last_path,
            device,
            model,
            ema,
            optimizer,
            scheduler,
            scaler,
            log_path,
        )
        first_epoch = saved_epoch + 1
        print(f"resuming from epoch {saved_epoch} ({last_path})", flush=True)
    else:
        if args.resume:
            print(
                f"--resume given but {last_path} does not exist: "
                "starting from scratch"
            )

        with open(log_path, "w", newline="") as file:
            csv.writer(file).writerow(LOG_FIELDS)

    start = time.time()

    for epoch in range(first_epoch, args.epochs + 1):
        epoch_start = time.time()
        train_loss, train_acc = train_one_epoch(
            args,
            train_model,
            model,
            ema,
            teacher,
            train_loader,
            optimizer,
            scheduler,
            scaler,
            criterion,
            device,
            dtype,
        )

        row = {
            "epoch": epoch,
            "lr": f"{optimizer.param_groups[0]['lr']:.5f}",
            "train_loss": f"{train_loss.item():.4f}",
            "train_acc": f"{train_acc.item():.4f}",
            "val_loss": "",
            "val_acc": "",
            "ema_val_acc": "",
            "epoch_time": f"{time.time() - epoch_start:.1f}",
        }

        if val_split is not None:
            val_images, val_labels = val_split
            logprobs = predict_logprobs(
                model, val_images, mean, std, device, tta=False, dtype=dtype
            )
            row["val_loss"] = f"{F.nll_loss(logprobs, val_labels).item():.4f}"
            row["val_acc"] = f"{accuracy(logprobs, val_labels):.4f}"

            if ema is not None:
                logprobs = predict_logprobs(
                    ema.module, val_images, mean, std, device, tta=False, dtype=dtype
                )
                row["ema_val_acc"] = f"{accuracy(logprobs, val_labels):.4f}"

        with open(log_path, "a", newline="") as file:
            csv.DictWriter(file, fieldnames=LOG_FIELDS).writerow(row)

        print(
            " | ".join(f"{key} {value}" for key, value in row.items() if value != ""),
            flush=True,
        )

        if args.save_every and epoch % args.save_every == 0 and epoch < args.epochs:
            save_checkpoint(
                last_path,
                epoch,
                model,
                ema,
                optimizer,
                scheduler,
                scaler,
                minutes_before + (time.time() - start) / 60,
            )

    minutes = minutes_before + (time.time() - start) / 60

    # Final evaluation: the raw and EMA weights are compared on the
    # validation images, with and without flip TTA.
    candidates = {"raw": model}

    if ema is not None:
        candidates["ema"] = ema.module

    metrics = {"params": n_params, "minutes": round(minutes, 1)}

    if val_split is not None:
        val_images, val_labels = val_split

        for tag, candidate in candidates.items():
            for tta in (False, True):
                logprobs = predict_logprobs(
                    candidate, val_images, mean, std, device, tta=tta, dtype=dtype
                )
                key = f"{tag}_val_acc_tta" if tta else f"{tag}_val_acc"
                metrics[key] = round(accuracy(logprobs, val_labels), 4)

        selected = max(candidates, key=lambda tag: metrics[f"{tag}_val_acc_tta"])
    else:
        # Without validation images there is nothing to compare, so the EMA
        # weights are used whenever they exist.
        selected = "ema" if ema is not None else "raw"

    metrics["selected"] = selected

    # Test predictions and the Kaggle submission
    test_logprobs = predict_logprobs(
        candidates[selected], test_images, mean, std, device, tta=True, dtype=dtype
    )
    np.save(os.path.join(run_dir, "test_logprobs.npy"), test_logprobs.numpy())
    write_submission(
        os.path.join(run_dir, "submission.csv"),
        test_ids,
        test_logprobs.argmax(1),
    )

    torch.save(
        {
            "model": args.model,
            "model_overrides": overrides,
            "args": vars(args),
            "state_dict": model.state_dict(),
            "ema_state_dict": ema.module.state_dict() if ema is not None else None,
            "selected": selected,
            "metrics": metrics,
        },
        os.path.join(run_dir, "model.pt"),
    )

    with open(os.path.join(run_dir, "metrics.json"), "w") as file:
        json.dump(metrics, file, indent=2)

    # The run is finished and model.pt holds the final weights, so the
    # resumable checkpoint is no longer needed.
    if os.path.exists(last_path):
        os.remove(last_path)

    append_summary(
        os.path.join(args.out_dir, "summary.csv"),
        {
            "name": args.name,
            "model": args.model,
            "params": n_params,
            "epochs": args.epochs,
            "batch_size": args.batch_size,
            "lr": args.lr,
            "wd": args.wd,
            "aug": args.aug,
            "mix": args.mix,
            "label_smoothing": args.label_smoothing,
            "ema": args.ema,
            "teacher": args.teacher or "",
            "val_size": args.val_size,
            "val_acc_tta": metrics.get(f"{selected}_val_acc_tta", ""),
            "selected": selected,
            "minutes": metrics["minutes"],
        },
    )

    print(f"\nNumber of trainable parameters: {n_params:,}")

    for key, value in metrics.items():
        print(f"{key}: {value}")

    print(
        f"submission written to {os.path.join(run_dir, 'submission.csv')} "
        f"(predictions from the {selected} weights, flip TTA)"
    )


if __name__ == "__main__":
    main(get_args())

In [ ]:
%%writefile prepare_data.py
"""Decode the competition PNG files once and save them as numpy arrays.

Decoding 50,000 small PNG files in every epoch would make data loading the
bottleneck, so it is done a single time and saved as:

    data/train.npz  images (50000, 32, 32, 3) uint8, labels (50000,) int64,
                    mean and std (3,) float32, per-channel values in [0, 1]
    data/test.npz   images (2000, 32, 32, 3) uint8, ids (2000,) int64

Usage:
    python prepare_data.py --src ../data --out data
"""
import argparse
import csv
import os

import numpy as np
from PIL import Image


def read_split(csv_path, image_dir, has_labels):
    with open(csv_path, newline="") as file:
        rows = list(csv.DictReader(file))

    images = np.empty((len(rows), 32, 32, 3), dtype=np.uint8)

    for index, row in enumerate(rows):
        with Image.open(os.path.join(image_dir, row["filename"])) as image:
            images[index] = np.asarray(image.convert("RGB"))

    ids = np.array([int(row["id"]) for row in rows], dtype=np.int64)

    if has_labels:
        labels = np.array([int(row["label"]) for row in rows], dtype=np.int64)
    else:
        labels = None

    return images, ids, labels


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--src", default="../data", help="folder with the Kaggle files"
    )
    parser.add_argument(
        "--out", default="data", help="where to write the .npz files"
    )
    args = parser.parse_args()
    os.makedirs(args.out, exist_ok=True)

    train_images, _, train_labels = read_split(
        os.path.join(args.src, "train_labels.csv"),
        os.path.join(args.src, "train_images"),
        True,
    )

    # Per-channel statistics of the training images, used for normalization.
    pixels = train_images.reshape(-1, 3).astype(np.float64) / 255.0
    mean, std = pixels.mean(axis=0), pixels.std(axis=0)

    np.savez(
        os.path.join(args.out, "train.npz"),
        images=train_images,
        labels=train_labels,
        mean=mean.astype(np.float32),
        std=std.astype(np.float32),
    )
    print(f"train: {train_images.shape}, class counts {np.bincount(train_labels).tolist()}")
    print(f"mean {mean.round(4).tolist()}  std {std.round(4).tolist()}")

    test_images, test_ids, _ = read_split(
        os.path.join(args.src, "test.csv"),
        os.path.join(args.src, "test_images"),
        False,
    )
    np.savez(os.path.join(args.out, "test.npz"), images=test_images, ids=test_ids)
    print(f"test:  {test_images.shape}")


if __name__ == "__main__":
    main()

## Decode the images once and check the parameter counts

In [ ]:
!python prepare_data.py --src {SOURCE_DIR} --out /tmp/data
!python models.py

## Runs

The Kaggle runs were done in batches, one notebook version per batch. `BATCH` picks the batch to run.
The runs are split across the two GPUs, progress is printed every 2 minutes and the full logs are in
`logs/`.

In [ ]:
BATCHES = {
    # Day 1: a quick baseline and the three architectures, same recipe.
    "architectures": [
        ("baseline", "--model resnet18_slim --epochs 30 --aug cutout"),
        ("arch_resnet18_slim", "--model resnet18_slim --epochs 60 --aug cutout"),
        ("arch_resnet14_wide", "--model resnet14_wide --epochs 60 --aug cutout"),
        ("arch_wrn28_3.5", "--model wrn28_3.5 --epochs 60 --aug cutout"),
    ],
    # Stronger augmentation at 60 epochs (same runs as plans/ablations.txt).
    "augmentation": [
        ("wrn_ta", "--model wrn28_3.5 --epochs 60 --aug ta+cutout"),
        ("wrn_aa", "--model wrn28_3.5 --epochs 60 --aug aa+cutout"),
        ("wrn_ta_mix", "--model wrn28_3.5 --epochs 60 --aug ta+cutout --mix both"),
        ("wide_ta", "--model resnet14_wide --epochs 60 --aug ta+cutout"),
    ],
    # AutoAugment with a long schedule.
    "long": [
        ("wrn_aa_200", "--model wrn28_3.5 --epochs 200 --aug aa+cutout"),
    ],
}
BATCH = "augmentation"
RUNS = BATCHES[BATCH]
COMMON = "--data-dir /tmp/data"

In [ ]:
def show_progress():
    # The last line of each log is the most recent epoch.
    for name, _ in RUNS:
        path = f"logs/{name}.out"
        lines = open(path).read().strip().splitlines() if os.path.exists(path) else []
        print(f"{name:20s} {lines[-1][:140] if lines else 'waiting'}")

    print("-" * 60, flush=True)


workers = max(1, (os.cpu_count() or 2) // n_gpus)
os.makedirs("logs", exist_ok=True)
processes = []

# Each GPU gets every n_gpus-th run and works through them one after another.
for gpu in range(n_gpus):
    gpu_runs = [run for index, run in enumerate(RUNS) if index % n_gpus == gpu]
    command = " ; ".join(
        f"python -u train.py --name {name} {arguments} {COMMON} --workers {workers} "
        f"> logs/{name}.out 2>&1"
        for name, arguments in gpu_runs
    )
    processes.append(
        subprocess.Popen(
            command,
            shell=True,
            env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)},
        )
    )
    print(f"GPU {gpu}: {[name for name, _ in gpu_runs]}")

while any(process.poll() is None for process in processes):
    time.sleep(120)
    show_progress()

show_progress()
print("all runs finished")

## Results

In [ ]:
summary = pd.read_csv("runs/summary.csv")
display(
    summary[["name", "model", "params", "epochs", "aug", "val_acc_tta", "selected", "minutes"]]
    .sort_values("val_acc_tta", ascending=False)
)

for name in summary["name"]:
    shutil.copy(f"runs/{name}/submission.csv", f"/kaggle/working/submission_{name}.csv")

# The run with the best validation accuracy becomes submission.csv.
best = summary.sort_values("val_acc_tta").iloc[-1]["name"]
shutil.copy(f"runs/{best}/submission.csv", "/kaggle/working/submission.csv")
print("submission.csv is from:", best)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name in summary["name"]:
    log = pd.read_csv(f"runs/{name}/log.csv")
    axes[0].plot(log["epoch"], log["train_loss"], label=name)
    axes[1].plot(log["epoch"], log["val_acc"], label=name)

axes[0].set(title="Training loss", xlabel="epoch")
axes[1].set(title="Validation accuracy", xlabel="epoch", ylim=(0.5, 1.0))

for axis in axes:
    axis.legend()
    axis.grid(alpha=0.3)

plt.tight_layout()
plt.show()